# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json
# Define the dataset URL
url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")


## 2. Data Overview
Review available record sets, fields, and their IDs.

In [ ]:
# List all available record sets with their @id and name
print("Available Record Sets:")
record_sets_info = []
for rs in metadata.record_sets:
    print(f"- @id: {rs['@id']} | Name: {rs['name'] if 'name' in rs else ''}")
    record_sets_info.append((rs['@id'], rs['name'] if 'name' in rs else ''))

if not record_sets_info:
    print("No record sets found in metadata. Trying to load records without discovery.")
else:
    print(f"Found {len(record_sets_info)} record sets.")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# Attempt to load all record sets data into DataFrames
import warnings
warnings.filterwarnings('ignore', category=FutureWarning)

dataframes = {}

auto_detected_record_set = None

# Try to extract all record sets by @id
if hasattr(metadata, 'record_sets') and metadata.record_sets:
    record_sets_list = [rs['@id'] for rs in metadata.record_sets]
else:
    # If not available as per Croissant, try typical default names (domain knowledge, fallback)
    record_sets_list = ['protein_table', 'table1', 'main', 'data']  # Try common names

# Scan through possible record_set IDs
loaded = False
for rs_id in record_sets_list:
    try:
        records = list(dataset.records(record_set=rs_id))
        if records:
            df = pd.DataFrame(records)
            dataframes[rs_id] = df
            auto_detected_record_set = rs_id
            print(f"Loaded record set: {rs_id} with columns: {df.columns.tolist()}")
            loaded = True
    except Exception as e:
        print(f"Could not load records for record_set {rs_id}: {str(e)}")

if not loaded:
    # Try to load the default (if only one record set is present)
    print("No explicit record sets loaded. Attempting to guess.")
    try:
        records = list(dataset.records())
        if records:
            df = pd.DataFrame(records)
            dataframes['default'] = df
            auto_detected_record_set = 'default'
            print(f"Loaded default record set with columns: {df.columns.tolist()}")
    except Exception as e:
        print(f"Could not load any records: {str(e)}")

if dataframes:
    example_rs_id = list(dataframes.keys())[0]
    print(f"Preview columns for the first record set: {example_rs_id}")
    print(dataframes[example_rs_id].columns.tolist())
    display(dataframes[example_rs_id].head())
else:
    print("No dataframes loaded. Check dataset structure.")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section should include operations like removing outliers, transforming data distributions, or grouping data by key attributes to prepare it for further analysis.

In [ ]:
import numpy as np

# Try to pick a suitable numeric field
df = dataframes.get(auto_detected_record_set)
if df is not None and df.shape[1] > 0:
    numeric_columns = df.select_dtypes(include=[np.number]).columns.tolist()
    print("Numeric columns detected:", numeric_columns)
    if numeric_columns:
        numeric_field = numeric_columns[0]  # Just pick the first
        threshold = df[numeric_field].mean() if df[numeric_field].count() > 0 else 10

        filtered_df = df[df[numeric_field] > threshold].copy()
        print(f"Filtered records with {numeric_field} > {threshold}:")
        print(filtered_df.head())

        # Normalization
        norm_col = f"{numeric_field}_normalized"
        filtered_df[norm_col] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / (filtered_df[numeric_field].std() if filtered_df[numeric_field].std() > 0 else 1)
        print(f"Normalized {numeric_field} for filtered records:")
        print(filtered_df[[numeric_field, norm_col]].head())

        # Try to group by a categorical (object/string) field
        object_columns = df.select_dtypes(include=['object', 'category']).columns.tolist()
        group_field = object_columns[0] if object_columns else None
        if group_field:
            grouped_df = filtered_df.groupby(group_field).mean(numeric_only=True)
            print(f"Grouped data by {group_field}:")
            print(grouped_df.head())
    else:
        print("No numeric fields available for basic EDA.")
else:
    print("No data to perform EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if df is not None and numeric_columns:
    # Histogram of the chosen numeric field
    plt.figure(figsize=(8, 4))
    sns.histplot(df[numeric_field].dropna(), bins=30, kde=True)
    plt.title(f'Distribution of {numeric_field}')
    plt.xlabel(numeric_field)
    plt.ylabel('Frequency')
    plt.show()

    # If another numeric field exists, make a scatterplot
    if len(numeric_columns) >= 2:
        plt.figure(figsize=(6, 6))
        sns.scatterplot(x=df[numeric_columns[0]], y=df[numeric_columns[1]])
        plt.xlabel(numeric_columns[0])
        plt.ylabel(numeric_columns[1])
        plt.title(f'Scatterplot of {numeric_columns[0]} vs {numeric_columns[1]}')
        plt.show()
else:
    print("Insufficient numeric data for visualization.")

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

In this notebook, we loaded the FAIR² dataset: *Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells* directly using the `mlcroissant` library. 

- We discovered available record sets (tables) and loaded their data into DataFrames for exploration.
- Performed initial EDA: filtered and normalized numeric columns and explored simple groupings.
- Visualized distributions of the key quantitative fields.

*Next steps*: With detailed knowledge of the schema (all entities are referenced by their `@id`), you can perform deeper domain-specific analyses, cross-reference protein IDs, or integrate with other omics data. For reproducibility, always reference columns and record sets by their `@id` as displayed in the metadata step.